# 2026 FIFA 월드컵 우승팀 예측 분석

이 노트북은 아래 4개 데이터를 사용해 2026 FIFA 월드컵 우승팀을 확률적으로 예측합니다.

- `results.csv`
- `shootouts.csv`
- `former_names.csv`
- `goalscorers.csv`

분석 흐름

1. CSV 파일 업로드
2. 데이터 전처리 및 국가명 정리
3. Elo Rating 기반 국가 전력 계산
4. 최근 경기 흐름 및 득점자 정보 반영
5. 경기 결과 예측 모델 학습
6. 2026 월드컵 조별리그 및 토너먼트 시뮬레이션
7. 우승 확률 결과 CSV 저장 및 다운로드


In [ ]:
# ============================================================
# 0. 라이브러리 설치 및 불러오기
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from collections import defaultdict, deque
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("라이브러리 불러오기 완료")


라이브러리 불러오기 완료


In [ ]:
# ============================================================
# 1. Colab 파일 업로드
# ============================================================

if IN_COLAB:
    print("CSV 파일 4개를 업로드해주세요.")
    print("필요 파일명: results.csv, shootouts.csv, former_names.csv, goalscorers.csv")
    uploaded = files.upload()

    print("\n업로드 완료 파일:")
    for filename in uploaded.keys():
        print("-", filename)
else:
    print("Colab 환경이 아닙니다. 현재 작업 폴더에 CSV 파일 4개가 있어야 합니다.")


CSV 파일 4개를 업로드해주세요.
필요 파일명: results.csv, shootouts.csv, former_names.csv, goalscorers.csv


Saving former_names.csv to former_names.csv
Saving goalscorers.csv to goalscorers.csv
Saving results.csv to results.csv
Saving shootouts.csv to shootouts.csv

업로드 완료 파일:
- former_names.csv
- goalscorers.csv
- results.csv
- shootouts.csv


In [ ]:
# ============================================================
# 2. 파일 경로 설정 및 존재 여부 확인
# ============================================================

results_path = "results.csv"
shootouts_path = "shootouts.csv"
former_names_path = "former_names.csv"
goalscorers_path = "goalscorers.csv"

required_files = [
    results_path,
    shootouts_path,
    former_names_path,
    goalscorers_path
]

missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    raise FileNotFoundError(
        "다음 파일이 없습니다: "
        + ", ".join(missing_files)
        + "\n파일명이 정확한지 확인해주세요."
    )

print("필수 파일 확인 완료")


필수 파일 확인 완료


In [ ]:
# ============================================================
# 3. 데이터 불러오기
# ============================================================

results = pd.read_csv(results_path)
shootouts = pd.read_csv(shootouts_path)
former_names = pd.read_csv(former_names_path)
goalscorers = pd.read_csv(goalscorers_path)

print("[데이터 크기 확인]")
print("results:", results.shape)
print("shootouts:", shootouts.shape)
print("former_names:", former_names.shape)
print("goalscorers:", goalscorers.shape)

print("\n[컬럼 확인]")
print("results:", results.columns.tolist())
print("shootouts:", shootouts.columns.tolist())
print("former_names:", former_names.columns.tolist())
print("goalscorers:", goalscorers.columns.tolist())


[데이터 크기 확인]
results: (49437, 9)
shootouts: (678, 5)
former_names: (36, 4)
goalscorers: (47601, 8)

[컬럼 확인]
results: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
shootouts: ['date', 'home_team', 'away_team', 'winner', 'first_shooter']
former_names: ['current', 'former', 'start_date', 'end_date']
goalscorers: ['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']


In [ ]:
# ============================================================
# 4. 날짜 및 기본 전처리
# ============================================================

results["date"] = pd.to_datetime(results["date"], errors="coerce")
shootouts["date"] = pd.to_datetime(shootouts["date"], errors="coerce")
goalscorers["date"] = pd.to_datetime(goalscorers["date"], errors="coerce")

results = results.dropna(
    subset=["date", "home_team", "away_team", "home_score", "away_score"]
).copy()

results["home_score"] = pd.to_numeric(results["home_score"], errors="coerce")
results["away_score"] = pd.to_numeric(results["away_score"], errors="coerce")
results = results.dropna(subset=["home_score", "away_score"]).copy()

results = results.sort_values("date").reset_index(drop=True)

if "neutral" not in results.columns:
    results["neutral"] = False

if "tournament" not in results.columns:
    results["tournament"] = "Unknown"

print("전처리 완료")
print(results.head())


전처리 완료
        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1 1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2 1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3 1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4 1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


In [ ]:
# ============================================================
# 5. 국가명 표기 정리
# ============================================================

name_map = {}

# former_names.csv 구조가 current / former 컬럼이라고 가정
if {"current", "former"}.issubset(former_names.columns):
    for _, row in former_names.iterrows():
        current = row["current"]
        former = row["former"]
        if pd.notna(current) and pd.notna(former):
            name_map[str(former).strip()] = str(current).strip()

# 수동 국가명 보정
manual_name_map = {
    "Korea Republic": "South Korea",
    "Republic of Korea": "South Korea",
    "United States": "USA",
    "United States of America": "USA",
    "Türkiye": "Turkey",
    "Czechia": "Czech Republic",
    "Côte d'Ivoire": "Ivory Coast",
    "Cote d'Ivoire": "Ivory Coast",
    "IR Iran": "Iran",
    "Curacao": "Curaçao",
    "DR Congo": "DR Congo",
    "Democratic Republic of Congo": "DR Congo",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
}

name_map.update(manual_name_map)


def normalize_team_name(team):
    # 국가명 표기 통일 함수
    if pd.isna(team):
        return team

    team = str(team).strip()
    return name_map.get(team, team)


for col in ["home_team", "away_team", "country"]:
    if col in results.columns:
        results[col] = results[col].apply(normalize_team_name)

for col in ["home_team", "away_team", "winner", "first_shooter"]:
    if col in shootouts.columns:
        shootouts[col] = shootouts[col].apply(normalize_team_name)

for col in ["home_team", "away_team", "team"]:
    if col in goalscorers.columns:
        goalscorers[col] = goalscorers[col].apply(normalize_team_name)

print("국가명 정리 완료")


국가명 정리 완료


In [ ]:
# ============================================================
# 6. 경기 결과 변수 생성
# ============================================================

def get_outcome(row):
    # 1: 홈팀 승, 0: 무승부, -1: 원정팀 승
    if row["home_score"] > row["away_score"]:
        return 1
    elif row["home_score"] < row["away_score"]:
        return -1
    else:
        return 0


results["outcome"] = results.apply(get_outcome, axis=1)
results["goal_diff"] = results["home_score"] - results["away_score"]
results["total_goals"] = results["home_score"] + results["away_score"]

print(results[["date", "home_team", "away_team", "home_score", "away_score", "outcome"]].head())


        date home_team away_team  home_score  away_score  outcome
0 1872-11-30  Scotland   England         0.0         0.0        0
1 1873-03-08   England  Scotland         4.0         2.0        1
2 1874-03-07  Scotland   England         2.0         1.0        1
3 1875-03-06   England  Scotland         2.0         2.0        0
4 1876-03-04  Scotland   England         3.0         0.0        1


In [ ]:
# ============================================================
# 7. 대회 중요도 가중치 생성
# ============================================================

def tournament_weight(tournament):
    # 친선경기보다 월드컵/예선/대륙컵 경기에 높은 가중치를 부여
    tournament = str(tournament).lower()

    if "fifa world cup" in tournament:
        return 3.0
    elif "uefa euro" in tournament:
        return 2.5
    elif "copa américa" in tournament or "copa america" in tournament:
        return 2.5
    elif "africa cup" in tournament:
        return 2.3
    elif "asian cup" in tournament:
        return 2.3
    elif "gold cup" in tournament:
        return 2.2
    elif "qualification" in tournament or "qualifying" in tournament:
        return 2.0
    elif "nations league" in tournament:
        return 1.8
    elif "friendly" in tournament:
        return 1.0
    else:
        return 1.2


results["tournament_weight"] = results["tournament"].apply(tournament_weight)

print(results[["tournament", "tournament_weight"]].drop_duplicates().head(20))


                              tournament  tournament_weight
0                               Friendly                1.0
29             British Home Championship                1.2
163                 Évence Coppée Trophy                1.2
174                         Muratti Vase                1.2
180                          Copa Lipton                1.2
196                          Copa Newton                1.2
237          Copa Premio Honor Argentino                1.2
238                        Olympic Games                1.2
320           Copa Premio Honor Uruguayo                1.2
385       Far Eastern Championship Games                1.2
450                            Copa Roca                1.2
480                         Copa América                2.5
567                   Inter-Allied Games                1.2
657                            Peace Cup                1.2
679      Open International Championship                1.2
738                         Soccer Ashes

In [ ]:
# ============================================================
# 8. Elo Rating 및 최근 경기 흐름 feature 생성
# ============================================================

BASE_ELO = 1500
K_FACTOR = 30
RECENT_N = 10

team_elo = defaultdict(lambda: BASE_ELO)

recent_points = defaultdict(lambda: deque(maxlen=RECENT_N))
recent_goals_for = defaultdict(lambda: deque(maxlen=RECENT_N))
recent_goals_against = defaultdict(lambda: deque(maxlen=RECENT_N))


def expected_score(elo_a, elo_b):
    # Elo 기대 승률 계산
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))


def points_from_match(score_for, score_against):
    # 승점 계산
    if score_for > score_against:
        return 3
    elif score_for == score_against:
        return 1
    else:
        return 0


def recent_avg(dictionary, team):
    # 최근 N경기 평균값 계산
    values = dictionary[team]
    if len(values) == 0:
        return 0
    return float(np.mean(values))


def update_elo(team_a, team_b, score_a, score_b, weight=1.0):
    # 경기 결과에 따라 Elo 점수 갱신
    elo_a = team_elo[team_a]
    elo_b = team_elo[team_b]

    exp_a = expected_score(elo_a, elo_b)
    exp_b = expected_score(elo_b, elo_a)

    if score_a > score_b:
        actual_a, actual_b = 1, 0
    elif score_a < score_b:
        actual_a, actual_b = 0, 1
    else:
        actual_a, actual_b = 0.5, 0.5

    goal_margin = abs(score_a - score_b)
    margin_multiplier = np.log(goal_margin + 1) + 1

    team_elo[team_a] += K_FACTOR * weight * margin_multiplier * (actual_a - exp_a)
    team_elo[team_b] += K_FACTOR * weight * margin_multiplier * (actual_b - exp_b)


feature_rows = []

for _, row in results.iterrows():
    home = row["home_team"]
    away = row["away_team"]

    home_elo_before = team_elo[home]
    away_elo_before = team_elo[away]

    feature_rows.append({
        "date": row["date"],
        "home_team": home,
        "away_team": away,

        "home_elo": home_elo_before,
        "away_elo": away_elo_before,
        "elo_diff": home_elo_before - away_elo_before,

        "neutral": int(bool(row["neutral"])),
        "tournament_weight": row["tournament_weight"],

        "home_recent_points": recent_avg(recent_points, home),
        "away_recent_points": recent_avg(recent_points, away),

        "home_recent_gf": recent_avg(recent_goals_for, home),
        "away_recent_gf": recent_avg(recent_goals_for, away),

        "home_recent_ga": recent_avg(recent_goals_against, home),
        "away_recent_ga": recent_avg(recent_goals_against, away),

        "outcome": row["outcome"]
    })

    update_elo(
        home,
        away,
        row["home_score"],
        row["away_score"],
        weight=row["tournament_weight"]
    )

    recent_points[home].append(points_from_match(row["home_score"], row["away_score"]))
    recent_points[away].append(points_from_match(row["away_score"], row["home_score"]))

    recent_goals_for[home].append(row["home_score"])
    recent_goals_for[away].append(row["away_score"])

    recent_goals_against[home].append(row["away_score"])
    recent_goals_against[away].append(row["home_score"])


model_df = pd.DataFrame(feature_rows)

print("[모델 학습용 데이터]")
print(model_df.head())
print("shape:", model_df.shape)


[모델 학습용 데이터]
        date home_team away_team     home_elo     away_elo   elo_diff  \
0 1872-11-30  Scotland   England  1500.000000  1500.000000   0.000000   
1 1873-03-08   England  Scotland  1500.000000  1500.000000   0.000000   
2 1874-03-07  Scotland   England  1468.520816  1531.479184 -62.958369   
3 1875-03-06   England  Scotland  1501.529501  1498.470499   3.059002   
4 1876-03-04  Scotland   England  1498.602563  1501.397437  -2.794873   

   neutral  tournament_weight  home_recent_points  away_recent_points  \
0        0                1.0            0.000000            0.000000   
1        0                1.0            1.000000            1.000000   
2        0                1.0            0.500000            2.000000   
3        0                1.0            1.333333            1.333333   
4        0                1.0            1.250000            1.250000   

   home_recent_gf  away_recent_gf  home_recent_ga  away_recent_ga  outcome  
0        0.000000        0.00000

In [ ]:
# ============================================================
# 9. goalscorers.csv 기반 공격력 보조 feature 생성
# ============================================================

recent_cutoff = results["date"].max() - pd.DateOffset(years=4)
recent_goals = goalscorers[goalscorers["date"] >= recent_cutoff].copy()

if "own_goal" in recent_goals.columns:
    recent_goals = recent_goals[recent_goals["own_goal"] == False].copy()

if "penalty" not in recent_goals.columns:
    recent_goals["penalty"] = False

if "scorer" not in recent_goals.columns:
    recent_goals["scorer"] = "Unknown"

if "team" in recent_goals.columns:
    scorer_stats = recent_goals.groupby("team").agg(
        recent_goal_records=("scorer", "count"),
        recent_unique_scorers=("scorer", "nunique"),
        recent_penalty_goals=("penalty", "sum")
    ).reset_index()
else:
    scorer_stats = pd.DataFrame(columns=[
        "team",
        "recent_goal_records",
        "recent_unique_scorers",
        "recent_penalty_goals"
    ])

scorer_stats["recent_penalty_rate"] = (
    scorer_stats["recent_penalty_goals"] /
    scorer_stats["recent_goal_records"].replace(0, np.nan)
).fillna(0)

print("[득점자 기반 보조 feature]")
print(scorer_stats.head())


[득점자 기반 보조 feature]
          team  recent_goal_records  recent_unique_scorers  \
0  Afghanistan                    5                      5   
1      Albania                   30                     15   
2      Algeria                   33                     15   
3      Andorra                   12                      8   
4       Angola                   19                      9   

   recent_penalty_goals  recent_penalty_rate  
0                     1             0.200000  
1                     4             0.133333  
2                     3             0.090909  
3                     1             0.083333  
4                     1             0.052632  


In [ ]:
# ============================================================
# 10. 승부차기 기록 보조 feature 생성
# ============================================================

if "winner" in shootouts.columns:
    shootout_win_count = shootouts.groupby("winner").size().reset_index(name="shootout_wins")
    shootout_win_count = shootout_win_count.rename(columns={"winner": "team"})
else:
    shootout_win_count = pd.DataFrame(columns=["team", "shootout_wins"])

print(shootout_win_count.head())


                  team  shootout_wins
0             Abkhazia              2
1              Algeria              7
2               Angola              7
3             Anguilla              1
4  Antigua and Barbuda              2


In [ ]:
# ============================================================
# 11. 모델 학습
# ============================================================

features = [
    "home_elo",
    "away_elo",
    "elo_diff",
    "neutral",
    "tournament_weight",
    "home_recent_points",
    "away_recent_points",
    "home_recent_gf",
    "away_recent_gf",
    "home_recent_ga",
    "away_recent_ga",
]

X = model_df[features]
y = model_df["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("[모델 평가]")
print("정확도:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred))


[모델 평가]
정확도: 0.5905
              precision    recall  f1-score   support

          -1       0.58      0.58      0.58      2863
           0       0.30      0.01      0.01      2301
           1       0.60      0.88      0.71      4709

    accuracy                           0.59      9873
   macro avg       0.49      0.49      0.43      9873
weighted avg       0.52      0.59      0.51      9873



In [ ]:
# ============================================================
# 12. 2026 FIFA 월드컵 참가국/조 편성
# ============================================================

worldcup_2026_groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["USA", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

teams_2026 = sorted([team for group in worldcup_2026_groups.values() for team in group])

print("[2026 월드컵 참가국 수]")
print(len(teams_2026))
print(teams_2026)


[2026 월드컵 참가국 수]
48
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'USA', 'Uruguay', 'Uzbekistan']


In [ ]:
# ============================================================
# 13. 참가국별 현재 전력 점수 계산
# ============================================================

scorer_lookup = scorer_stats.set_index("team").to_dict("index") if len(scorer_stats) > 0 else {}
shootout_lookup = shootout_win_count.set_index("team")["shootout_wins"].to_dict() if len(shootout_win_count) > 0 else {}

team_strength = []

for team in teams_2026:
    elo = team_elo[team]
    recent_pts = recent_avg(recent_points, team)
    recent_gf = recent_avg(recent_goals_for, team)
    recent_ga = recent_avg(recent_goals_against, team)

    scorer_info = scorer_lookup.get(team, {})
    unique_scorers = scorer_info.get("recent_unique_scorers", 0)
    recent_goal_records = scorer_info.get("recent_goal_records", 0)

    shootout_wins = shootout_lookup.get(team, 0)

    # 종합 전력 점수
    strength_score = (
        elo
        + recent_pts * 20
        + recent_gf * 15
        - recent_ga * 10
        + unique_scorers * 1.5
        + recent_goal_records * 0.05
        + shootout_wins * 0.3
    )

    team_strength.append({
        "team": team,
        "elo": round(elo, 2),
        "recent_points_avg": round(recent_pts, 3),
        "recent_goals_for_avg": round(recent_gf, 3),
        "recent_goals_against_avg": round(recent_ga, 3),
        "recent_unique_scorers": unique_scorers,
        "recent_goal_records": recent_goal_records,
        "shootout_wins": shootout_wins,
        "strength_score": round(strength_score, 3)
    })

strength_df = pd.DataFrame(team_strength).sort_values(
    "strength_score",
    ascending=False
).reset_index(drop=True)

print("[2026 참가국 전력 순위 TOP 20]")
display(strength_df.head(20))


[2026 참가국 전력 순위 TOP 20]


,team,elo,recent_points_avg,recent_goals_for_avg,recent_goals_against_avg,recent_unique_scorers,recent_goal_records,shootout_wins,strength_score
0,Spain,2466.46,2.2,2.7,0.5,29,98,7,2596.459
1,France,2294.99,2.5,2.4,0.8,25,82,5,2416.092
2,England,2292.54,2.2,2.2,0.5,28,79,4,2411.687
3,Turkey,2261.11,2.5,2.5,1.3,28,67,1,2381.257
4,Norway,2267.66,2.3,3.1,0.7,15,68,0,2379.060
5,Argentina,2259.65,2.5,2.3,0.3,15,55,15,2370.898
6,Germany,2221.78,2.7,2.8,0.8,23,67,6,2349.428
7,Ecuador,2252.11,1.6,0.9,0.5,9,22,2,2307.814
8,Morocco,2185.29,2.4,2.2,0.3,16,41,6,2291.143
9,Netherlands,2157.20,2.1,2.8,0.7,26,95,2,2278.549


In [ ]:
# ============================================================
# 14. 경기 예측 함수
# ============================================================

def get_team_feature(team):
    # 특정 국가의 현재 전력 feature 반환
    return {
        "elo": team_elo[team],
        "recent_points": recent_avg(recent_points, team),
        "recent_gf": recent_avg(recent_goals_for, team),
        "recent_ga": recent_avg(recent_goals_against, team)
    }


def predict_match_proba(team_a, team_b, neutral=True, tournament_weight_value=3.0):
    # 두 팀의 경기 결과 확률 예측
    # 반환값: team_a 승리 확률, 무승부 확률, team_b 승리 확률
    a = get_team_feature(team_a)
    b = get_team_feature(team_b)

    row = pd.DataFrame([{
        "home_elo": a["elo"],
        "away_elo": b["elo"],
        "elo_diff": a["elo"] - b["elo"],
        "neutral": int(neutral),
        "tournament_weight": tournament_weight_value,
        "home_recent_points": a["recent_points"],
        "away_recent_points": b["recent_points"],
        "home_recent_gf": a["recent_gf"],
        "away_recent_gf": b["recent_gf"],
        "home_recent_ga": a["recent_ga"],
        "away_recent_ga": b["recent_ga"],
    }])

    proba = model.predict_proba(row)[0]
    class_map = dict(zip(model.classes_, proba))

    team_a_win = class_map.get(1, 0)
    draw = class_map.get(0, 0)
    team_b_win = class_map.get(-1, 0)

    total = team_a_win + draw + team_b_win
    if total == 0:
        return 0.45, 0.10, 0.45

    return team_a_win / total, draw / total, team_b_win / total


def simulate_match(team_a, team_b, allow_draw=False):
    # 경기 시뮬레이션
    # 조별리그는 무승부 허용, 토너먼트는 무승부 불가
    p_a, p_draw, p_b = predict_match_proba(team_a, team_b)

    if allow_draw:
        result = np.random.choice(
            [team_a, "DRAW", team_b],
            p=np.array([p_a, p_draw, p_b]) / (p_a + p_draw + p_b)
        )
        return result

    else:
        elo_a = team_elo[team_a]
        elo_b = team_elo[team_b]

        elo_prob_a = expected_score(elo_a, elo_b)

        final_p_a = p_a + p_draw * elo_prob_a
        final_p_b = p_b + p_draw * (1 - elo_prob_a)

        final_p_a = final_p_a / (final_p_a + final_p_b)
        final_p_b = 1 - final_p_a

        winner = np.random.choice([team_a, team_b], p=[final_p_a, final_p_b])
        return winner

print("경기 예측 함수 생성 완료")


경기 예측 함수 생성 완료


In [ ]:
# ============================================================
# 15. 조별리그 시뮬레이션 함수
# ============================================================

strength_lookup = strength_df.set_index("team")["strength_score"].to_dict()


def simulate_group(group_teams):
    # 한 조의 조별리그 시뮬레이션
    table = {
        team: {
            "team": team,
            "points": 0,
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "strength_score": strength_lookup.get(team, BASE_ELO)
        }
        for team in group_teams
    }

    for i in range(len(group_teams)):
        for j in range(i + 1, len(group_teams)):
            team_a = group_teams[i]
            team_b = group_teams[j]

            result = simulate_match(team_a, team_b, allow_draw=True)

            if result == team_a:
                table[team_a]["points"] += 3
                table[team_a]["wins"] += 1
                table[team_b]["losses"] += 1

            elif result == team_b:
                table[team_b]["points"] += 3
                table[team_b]["wins"] += 1
                table[team_a]["losses"] += 1

            else:
                table[team_a]["points"] += 1
                table[team_b]["points"] += 1
                table[team_a]["draws"] += 1
                table[team_b]["draws"] += 1

    group_table = pd.DataFrame(table.values())

    group_table = group_table.sort_values(
        ["points", "strength_score"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return group_table


def simulate_group_stage():
    # 전체 조별리그 시뮬레이션
    # 각 조 1, 2위 + 3위 중 상위 8개 팀이 32강 진출
    qualified = []
    third_places = []
    group_results = {}

    for group_name, teams in worldcup_2026_groups.items():
        group_table = simulate_group(teams)
        group_table["group"] = group_name
        group_results[group_name] = group_table

        qualified.extend(group_table.iloc[:2]["team"].tolist())
        third_places.append(group_table.iloc[2])

    third_df = pd.DataFrame(third_places)

    best_thirds = third_df.sort_values(
        ["points", "strength_score"],
        ascending=[False, False]
    ).head(8)

    qualified.extend(best_thirds["team"].tolist())

    return qualified, group_results

print("조별리그 시뮬레이션 함수 생성 완료")


조별리그 시뮬레이션 함수 생성 완료


In [ ]:
# ============================================================
# 16. 토너먼트 시뮬레이션 함수
# ============================================================

def simulate_knockout(qualified_teams):
    # 32강 이후 토너먼트 시뮬레이션
    # 실제 대진표가 아닌 간이 대진 방식:
    # 전력 순으로 정렬한 뒤 강팀 vs 약팀 매칭
    teams = sorted(
        qualified_teams,
        key=lambda x: strength_lookup.get(x, BASE_ELO),
        reverse=True
    )

    while len(teams) > 1:
        next_round = []

        for i in range(len(teams) // 2):
            team_a = teams[i]
            team_b = teams[-(i + 1)]

            winner = simulate_match(team_a, team_b, allow_draw=False)
            next_round.append(winner)

        teams = sorted(
            next_round,
            key=lambda x: strength_lookup.get(x, BASE_ELO),
            reverse=True
        )

    return teams[0]


def simulate_worldcup_once():
    # 월드컵 1회 전체 시뮬레이션
    qualified, group_results = simulate_group_stage()
    champion = simulate_knockout(qualified)
    return champion

print("토너먼트 시뮬레이션 함수 생성 완료")


토너먼트 시뮬레이션 함수 생성 완료


In [ ]:
# ============================================================
# 17. 몬테카를로 시뮬레이션 실행
# ============================================================

N_SIMULATIONS = 5000

champion_counts = defaultdict(int)

print("[월드컵 시뮬레이션 실행 중]")
print("반복 횟수:", N_SIMULATIONS)

for i in range(N_SIMULATIONS):
    champion = simulate_worldcup_once()
    champion_counts[champion] += 1

champion_df = pd.DataFrame([
    {
        "team": team,
        "champion_count": count,
        "champion_probability": count / N_SIMULATIONS,
        "champion_probability_percent": round((count / N_SIMULATIONS) * 100, 3)
    }
    for team, count in champion_counts.items()
]).sort_values("champion_probability", ascending=False).reset_index(drop=True)

print("[2026 FIFA 월드컵 우승 확률 TOP 20]")
display(champion_df.head(20))


[월드컵 시뮬레이션 실행 중]
반복 횟수: 5000
[2026 FIFA 월드컵 우승 확률 TOP 20]


,team,champion_count,champion_probability,champion_probability_percent
0,Spain,1521,0.3042,30.42
1,France,775,0.1550,15.50
2,England,669,0.1338,13.38
3,Turkey,422,0.0844,8.44
4,Norway,413,0.0826,8.26
5,Argentina,315,0.0630,6.30
6,Ecuador,227,0.0454,4.54
7,Germany,192,0.0384,3.84
8,Morocco,73,0.0146,1.46
9,Netherlands,69,0.0138,1.38


In [ ]:
# ============================================================
# 18. 결과 저장 및 다운로드
# ============================================================

strength_output = "worldcup_2026_team_strength.csv"
champion_output = "worldcup_2026_champion_prediction.csv"

strength_df.to_csv(strength_output, index=False, encoding="utf-8-sig")
champion_df.to_csv(champion_output, index=False, encoding="utf-8-sig")

print("[저장 완료]")
print("-", strength_output)
print("-", champion_output)

if IN_COLAB:
    files.download(strength_output)
    files.download(champion_output)


[저장 완료]
- worldcup_2026_team_strength.csv
- worldcup_2026_champion_prediction.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# 19. 최종 우승 예측 TOP 10 확인
# ============================================================

top10 = champion_df.head(10)[[
    "team",
    "champion_probability_percent",
    "champion_count"
]]

display(top10)

print("분석 완료")


,team,champion_probability_percent,champion_count
0,Spain,30.42,1521
1,France,15.50,775
2,England,13.38,669
3,Turkey,8.44,422
4,Norway,8.26,413
5,Argentina,6.30,315
6,Ecuador,4.54,227
7,Germany,3.84,192
8,Morocco,1.46,73
9,Netherlands,1.38,69


분석 완료
